In [1]:
import pandas as pd
import torch

data = pd.read_parquet("../data/external/robust04.parquet")

In [2]:
data = pd.read_parquet("../data/external/robust04.parquet")
topic_sample = data[data.query_id == "301"].sample(frac=1.0)
split = int(len(topic_sample) * 0.8)
train = topic_sample.iloc[:split].loc[:, ["query", "doc", "relevance"]]
test = topic_sample.iloc[split:].loc[:, ["query", "doc", "relevance"]]


In [3]:
split = int(len(topic_sample) * 0.8)
train = topic_sample.iloc[:split]
test = topic_sample.iloc[split:]

In [9]:
from torch.utils.data import IterableDataset
from typing import Iterator
from lightning_ir import RankSample


class Dataset(IterableDataset):
    def __init__(self, data: pd.DataFrame) -> None:
        self.data = data

    def prepare_data(self):
        pass

    def __iter__(self) -> Iterator[RankSample]:
        for idx, sample in self.data.iterrows():
            yield RankSample(sample.query_id, sample.query,[ sample.doc_id], [sample.doc], torch.Tensor([sample.relevance]))

In [13]:
from torch.optim import AdamW
from torch.nn.functional import mse_loss
from peft import LoraConfig
from lightning_ir.models import MonoConfig
from lightning_ir import CrossEncoderModule, LightningIRDataModule, LightningIRTrainer
from lightning_ir.loss import ScoringLossFunction


class MSELoss(ScoringLossFunction):
    def compute_loss(self, output, batch):
        return mse_loss(output.scores, batch.targets.squeeze(-1), reduction="mean")


module = CrossEncoderModule(
    model_name_or_path="cross-encoder/ms-marco-MiniLM-L12-v2",
    config=MonoConfig(
        query_length=32,
        doc_length=512,
        scoring_strategy="mono",
        use_adapter=True, # Enable adapter use
        adapter_config=LoraConfig( # Specify the adapter config
            r=16,
            lora_alpha=32,
            target_modules=["query", "key", "value"],
            lora_dropout=0.1
        ),
    ),
    loss_functions=[MSELoss()],
    evaluation_metrics=[]
)
module.set_optimizer(AdamW, lr=1e-6)


data_module = LightningIRDataModule(
    train_dataset=Dataset(train),
    train_batch_size=8,
    #inference_datasets=[Dataset(test)],
    #inference_batch_size=8,
)
data_module.prepare_data()


trainer = LightningIRTrainer(max_epochs=2, accumulate_grad_batches=8)
trainer.fit(module, data_module)

module.save_pretrained("./topic-301/")

You are using a model of type bert to instantiate a model of type mono. This is not supported for all configurations of models and can yield errors.
Some weights of MonoBertModel were not initialized from the model checkpoint at cross-encoder/ms-marco-MiniLM-L12-v2 and are newly initialized: ['linear.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'MonoBertTokenizerFast'.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/gienapp/Gits/ecir26-topic-adapters-as-judges/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/gienapp/Gits/ecir26-topic-adapters-as-judges/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

You are using a model of type bert to instantiate a model of type mono. This is not supported for all configurations of models and can yield errors.


Validation: |          | 0/? [00:00<?, ?it/s]

You are using a model of type bert to instantiate a model of type mono. This is not supported for all configurations of models and can yield errors.
`Trainer.fit` stopped: `max_epochs=2` reached.
You are using a model of type bert to instantiate a model of type mono. This is not supported for all configurations of models and can yield errors.


In [16]:
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("./topic-301/")
tokenizer = AutoTokenizer.from_pretrained("./topic-301/")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'CrossEncoderTokenizer'. 
The class this function is called from is 'CrossEncoderBertTokenizerFast'.


In [22]:
for _, (query_id, doc_id, query, doc, relevance) in test.iterrows():
    sample = tokenizer(query + " " + doc, return_tensors="pt", truncation=True, max_length=512)
    pred = model(**sample)
    break

In [23]:


trainer.test(datamodule=test)

/Users/gienapp/Gits/ecir26-topic-adapters-as-judges/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:149: `.test(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.test(ckpt_path='best')` to use the best model or `.test(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at /Users/gienapp/Gits/ecir26-topic-adapters-as-judges/notebooks/lightning_logs/version_6/checkpoints/epoch=1-step=38.ckpt
Loaded model weights from the checkpoint at /Users/gienapp/Gits/ecir26-topic-adapters-as-judges/notebooks/lightning_logs/version_6/checkpoints/epoch=1-step=38.ckpt


MisconfigurationException: `test_dataloader` must be implemented to be used with the Lightning Trainer